# Chapter 10 tutorial — Part 3: estimating variances, power, and adequacy

This notebook covers:

- sample size for estimating variances and standard deviations,
- the chi-square distribution connection,
- why estimating variability needs many observations,
- power of statistical tests,
- the relation between power, confidence intervals, and OC curves.

## Mental model

Estimating a mean is like locating the center of a cloud. Estimating a variance is like estimating the cloud's width. Width is harder: small samples often give unstable estimates of variability.

In [ ]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import binom, hypergeom, norm, chi2
from scipy.optimize import brentq

pd.set_option("display.precision", 4)

## 1. Variance estimation and the chi-square distribution

For a normal population with true variance $\sigma^2$, a sample variance $S^2$ based on $n = N-1$ degrees of freedom satisfies

$$
\frac{nS^2}{\sigma^2} \sim \chi_n^2.
$$

This is the core fact behind confidence intervals for $\sigma$ and sample-size planning for reproducibility studies.

In [ ]:
xs = np.linspace(0, 40, 500)
plt.figure()
for df in [3, 6, 12, 30]:
    plt.plot(xs, chi2.pdf(xs, df), label=f"df={df}")
plt.xlabel(r"$\chi^2$ value")
plt.ylabel("density")
plt.title("Chi-square distributions are skewed, especially for small df")
plt.legend()
plt.show()

## 2. Probability interval for $S/\sigma$

For $n=12$ degrees of freedom, the 5th and 95th percentiles of $\chi^2_{12}$ are approximately 5.226 and 21.026. Therefore, with 90% probability,

$$
5.226 < \frac{12S^2}{\sigma^2} < 21.026.
$$

Dividing by 12:

$$
0.4355 < \frac{S^2}{\sigma^2} < 1.7552.
$$

Taking square roots:

$$
0.66 < \frac{S}{\sigma} < 1.32.
$$

Mental model: even with 13 observations ($n=12$ df), the sample standard deviation can easily be about 34% low or 32% high.

In [ ]:
n = 12
q_low = chi2.ppf(0.05, n)
q_high = chi2.ppf(0.95, n)
ratio_var_low = q_low / n
ratio_var_high = q_high / n
ratio_sd_low = math.sqrt(ratio_var_low)
ratio_sd_high = math.sqrt(ratio_var_high)

pd.DataFrame([{
    "df n": n,
    "chi2 5%": q_low,
    "chi2 95%": q_high,
    "low S^2/sigma^2": ratio_var_low,
    "high S^2/sigma^2": ratio_var_high,
    "low S/sigma": ratio_sd_low,
    "high S/sigma": ratio_sd_high,
}])

## 3. Confidence interval for $\sigma$

From

$$
a < \frac{S}{\sigma} < b,
$$

we invert to get

$$
\frac{S}{b} < \sigma < \frac{S}{a}.
$$

Equivalently, for confidence level $1-\alpha$,

$$
\sqrt{\frac{nS^2}{\chi^2_{1-\alpha/2,n}}}
< \sigma <
\sqrt{\frac{nS^2}{\chi^2_{\alpha/2,n}}}.
$$

In [ ]:
def ci_sigma(sample_sd, df, confidence=0.90):
    alpha = 1 - confidence
    q_low = chi2.ppf(alpha/2, df)
    q_high = chi2.ppf(1 - alpha/2, df)
    lower = math.sqrt(df * sample_sd**2 / q_high)
    upper = math.sqrt(df * sample_sd**2 / q_low)
    return lower, upper

for s in [0.25, 0.43, 1.00]:
    lo, hi = ci_sigma(s, df=12, confidence=0.90)
    print(f"S={s:.2f}: 90% CI for sigma = ({lo:.3f}, {hi:.3f})")

## 4. Sample size for estimating a standard deviation

Suppose we want the sample standard deviation to be within 20% of the true standard deviation:

$$
0.8 < \frac{S}{\sigma} < 1.2.
$$

Squaring gives

$$
0.64 < \frac{S^2}{\sigma^2} < 1.44.
$$

Using the chi-square relation, this is

$$
P\left(n(0.8)^2 < \chi_n^2 < n(1.2)^2\right).
$$

We can search for the smallest $n$ that makes this probability at least the desired confidence level.

In [ ]:
def prob_sd_within(df, rel_error):
    lo = df * (1 - rel_error)**2
    hi = df * (1 + rel_error)**2
    return chi2.cdf(hi, df) - chi2.cdf(lo, df)

def min_sample_size_for_sd(rel_error=0.20, confidence=0.90, max_df=1000):
    for df in range(1, max_df + 1):
        if prob_sd_within(df, rel_error) >= confidence:
            return df + 1, df, prob_sd_within(df, rel_error)
    raise ValueError("increase max_df")

rows = []
for confidence in [0.80, 0.90, 0.95]:
    N, df, p = min_sample_size_for_sd(rel_error=0.20, confidence=confidence)
    rows.append({"confidence": confidence, "required N": N, "df": df, "achieved probability": p})
pd.DataFrame(rows)

The chapter's hand-table calculation gives an order-of-magnitude answer around dozens of observations for $\pm20\%$ accuracy. Exact quantiles and table conventions can shift the integer by several observations. The design lesson is robust: **small samples are unreliable for estimating variability**.

In [ ]:
ns = np.arange(2, 151)
probs = [prob_sd_within(n-1, 0.20) for n in ns]
plt.figure()
plt.plot(ns, probs)
plt.axhline(0.90, linestyle="--")
plt.axhline(0.95, linestyle="--")
plt.xlabel("sample size N")
plt.ylabel(r"P($0.8 < S/\sigma < 1.2$)")
plt.title("Many replicates are needed to estimate standard deviation precisely")
plt.show()

## 5. Approximate normal method for $S/\sigma$

For large $n$, the sample standard deviation is approximately normal with

$$
E[S] \approx \sigma,
\qquad
\mathrm{Var}(S) \approx \frac{\sigma^2}{2n}.
$$

Therefore

$$
\frac{S}{\sigma} \approx N\left(1, \frac{1}{2n}\right).
$$

For a desired relative half-width $e$ and confidence $1-\alpha$,

$$
e \approx z_{1-\alpha/2}\frac{1}{\sqrt{2n}},
$$

so

$$
n \approx \frac{1}{2}\left(\frac{z_{1-\alpha/2}}{e}\right)^2.
$$

In [ ]:
def approx_N_for_sd(rel_error=0.20, confidence=0.95):
    alpha = 1 - confidence
    z = norm.ppf(1 - alpha/2)
    df_cont = 0.5 * (z / rel_error) ** 2
    return df_cont + 1, math.ceil(df_cont + 1)

pd.DataFrame([
    {"confidence": c, "approx continuous N": approx_N_for_sd(0.20, c)[0], "approx integer N": approx_N_for_sd(0.20, c)[1]}
    for c in [0.90, 0.95]
])

## 6. Power of statistical tests

A significance level controls what happens when the null hypothesis is true:

$$
\alpha = P(\text{reject } H_0 \mid H_0 \text{ true}).
$$

Power controls what happens when the null is false:

$$
\mathrm{power}(\theta) = P(\text{reject } H_0 \mid \theta \text{ is true}).
$$

Mental model: significance is the **false-alarm policy**; power is the **detection capability**.

For a one-sided $z$ test of

$$
H_0: \mu = 0 \quad \text{versus} \quad H_1: \mu > 0,
$$

with known $\sigma$, reject when

$$
\bar X > z_{1-\alpha}\frac{\sigma}{\sqrt N}.
$$

If the true mean is $\mu = \delta$, then

$$
\mathrm{power}(\delta)
= 1 - \Phi\left(z_{1-\alpha} - \frac{\delta\sqrt N}{\sigma}\right).
$$

In [ ]:
def one_sided_z_power(delta, N, sigma=1.0, alpha=0.05):
    zcrit = norm.ppf(1 - alpha)
    return 1 - norm.cdf(zcrit - delta * math.sqrt(N) / sigma)

deltas = np.linspace(0, 1.5, 400)
plt.figure()
for N in [5, 10, 25, 100]:
    plt.plot(deltas, [one_sided_z_power(d, N) for d in deltas], label=f"N={N}")
plt.axhline(0.80, linestyle="--")
plt.xlabel(r"effect size $\delta/\sigma$")
plt.ylabel("power")
plt.title("Power increases with sample size and effect size")
plt.legend()
plt.show()

## 7. Relation between OC curves and power curves

For acceptance sampling, an OC curve gives

$$
P(\text{accept lot at quality } p).
$$

For a test, a power curve gives

$$
P(\text{reject null at true parameter } \theta).
$$

They are two sides of the same design question:

- OC curve: probability of **accepting** as quality changes.
- Power curve: probability of **detecting/rejecting** as departure from the null grows.

If acceptance means “do not detect a problem,” then

$$
\text{OC} \approx 1 - \text{power}.
$$

In [ ]:
# Show 1 - power for the same z-test as an OC-like curve.
plt.figure()
for N in [5, 10, 25, 100]:
    plt.plot(deltas, [1 - one_sided_z_power(d, N) for d in deltas], label=f"N={N}")
plt.xlabel(r"departure from null, $\delta/\sigma$")
plt.ylabel("probability of not rejecting")
plt.title("OC-like view: probability of not detecting a departure")
plt.legend()
plt.show()

## 8. Confidence interval length as an adequacy criterion

The chapter emphasizes that confidence interval length is often more directly useful than a formal null hypothesis.

For a mean with known $\sigma$ and 95% confidence,

$$
\text{CI length} = 2(1.96)\frac{\sigma}{\sqrt N}.
$$

Insufficient sample size produces long intervals, which means many materially different parameter values remain “acceptable” because the experiment was too weak to distinguish them.

In [ ]:
sigma = 1.0
Ns = np.arange(2, 301)
ci_lengths = 2 * norm.ppf(0.975) * sigma / np.sqrt(Ns)

plt.figure()
plt.plot(Ns, ci_lengths)
plt.xlabel("sample size N")
plt.ylabel("95% CI length for mean")
plt.title("Increasing N narrows the set of acceptable parameter values")
plt.show()

## 9. Final design checklist

Before collecting data, write down:

1. What is the target: decision, mean, variance, model choice, or process control?
2. What error is tolerable: $\alpha$, $\beta$, CI length, or relative error?
3. What variability estimate is available before the experiment?
4. What sample size follows from the statistical design?
5. What does that sample size cost?
6. Is the sample size adequate for the decision, not merely adequate for producing a $p$-value?

Mental model: sample size planning is asking whether the experiment has enough information to achieve its aim.